# Demo 07 : Cost

Run the cells in order and watch.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · a-model-per-agent

We assign a model instance to each specialist and measure its usage. Both use the course model in this comparison. Separate instances allow different budgets; they share the setup cell's limiter. We compare the tokens of a compact handoff with those of a full transcript. We do not measure latency.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool

DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))


@tool
def part_stock(part_no: str) -> str:
    """Look up a Veenhaven Bus part's stock and depot by its part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no = ?",
                          (part_no.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Read the meaning, first action and replacement condition for a bus error code."""
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error_code.upper() + " |")) or "No matching code."


QUESTION = "How many CE-CF-02 cooling fans are in stock and at which depot?"
SYSTEM = "Support Veenhaven Bus mechanics. Use the tools for facts. Preserve stock and depot names. Be brief."
LIMIT = {"recursion_limit": 20}


def ask(text):
    return {"messages": [{"role": "user", "content": text}]}


def show(result):
    for message in result["messages"]:
        if isinstance(message, dict):
            kind, content = message["type"], message.get("content", "")
            calls = message.get("tool_calls", [])
        else:
            kind, content, calls = message.type, message.content, getattr(message, "tool_calls", [])
        if kind in ("ai", "tool"):
            print(kind, ":", str(content)[:700], calls or "")

In [3]:
from langchain_core.callbacks import get_usage_metadata_callback

parts_model = make_llm(max_tokens=300)
summary_model = make_llm(max_tokens=200)
specialist = create_agent(parts_model, tools=[part_stock], system_prompt=SYSTEM)
with get_usage_metadata_callback() as usage:
    out = specialist.invoke(ask(QUESTION), LIMIT)
print("specialist :", usage.usage_metadata)
show(out)
full = "\n".join(str(m.content) + str(getattr(m, "tool_calls", [])) for m in out["messages"])
compact = str(out["messages"][-1].content)
for label, context in (("transcript", full), ("summary", compact)):
    with get_usage_metadata_callback() as usage:
        response = summary_model.invoke("Answer this bus stock question from the findings. " + QUESTION + "\n" + context)
    print(label, ":", usage.usage_metadata)
    print("answer :", response.content)

specialist : {'openai/gpt-6-luna': {'total_tokens': 289, 'input_tokens': 244, 'output_token_details': {'audio': 0, 'reasoning': 0}, 'output_tokens': 45, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}}}
ai :  [{'name': 'part_stock', 'args': {'part_no': 'CE-CF-02'}, 'id': 'call_rs2ddJ2EEcSaodrHzWveXaEY', 'type': 'tool_call'}]
tool : [["CE-CF-02", 6, "Rietpoort"]] 
ai : There are 6 CE-CF-02 cooling fans in stock at Rietpoort. 


transcript : {'openai/gpt-6-luna': {'input_tokens': 138, 'output_tokens': 24, 'total_tokens': 162, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}
answer : There are 6 CE-CF-02 cooling fans in stock at the Rietpoort depot.


summary : {'openai/gpt-6-luna': {'input_tokens': 50, 'output_tokens': 24, 'total_tokens': 74, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}
answer : There are 6 CE-CF-02 cooling fans in stock at the Rietpoort depot.


## b · prompt-caching

We repeat a stable bus-manual prefix with a changing question. Provider-reported cache reads can be zero; eligibility and retention depend on routing and the provider. Prompt caching still sends a request, unlike an application response cache.

In [4]:
from langchain.messages import SystemMessage

prefix = "You support Veenhaven Bus. Answer briefly from these manuals.\n" + "\n".join(
    p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md")))
for question in ("What is the CE-12 cooling fan interval?", "What is the CE-12 cooling fan interval on firmware 5.3?"):
    message = llm.invoke([SystemMessage(prefix), HumanMessage(question)])
    details = (message.usage_metadata or {}).get("input_token_details", {})
    print("usage :", message.usage_metadata)
    print("cache read :", details.get("cache_read", 0))
    print("answer :", message.content)

usage : {'input_tokens': 4805, 'output_tokens': 42, 'total_tokens': 4847, 'input_token_details': {'audio': 0, 'cache_read': 4802, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
cache read : 4802
answer : For CE-12 firmware **5.2 or later**, replace the traction cooling fan every **18 months**. Below firmware 5.2, the interval remains **12 months**.


usage : {'input_tokens': 4811, 'output_tokens': 24, 'total_tokens': 4835, 'input_token_details': {'audio': 0, 'cache_read': 4808, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
cache read : 4808
answer : Replace the CE-12 traction cooling fan every **18 months** on firmware 5.3.
